# 04 — GRPO Reinforcement Learning

Fine-tune with **Group Relative Policy Optimization** using `judger.auto_judge()`
as a binary verifiable reward — no separate reward model needed.

For each training step: generate G completions per question, score each with the
judger, compute group-normalized advantages, apply policy gradient + KL penalty.

**Prerequisites:** run `02_inference.ipynb` on the full public set, then optionally
`03_qlora_finetune.ipynb`. Set `ADAPTER_PATH` below accordingly.


In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# No-op when running locally.  On Colab: installs packages, mounts Drive.
import sys

try:
    import google.colab  # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

if _IS_COLAB:
    print("Colab detected — installing packages (takes ~3-5 min)...")
    import subprocess
    pkgs = [
        "vllm>=0.6.0", "transformers>=4.45.0", "bitsandbytes>=0.43.0",
        "peft>=0.12.0", "trl>=0.12.0", "datasets", "accelerate>=0.34.0",
        "sympy>=1.12", "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("Packages installed. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted. Upload public.jsonl / private.jsonl to")
    print("  MyDrive/math-qa-llm/data/raw/  before running further cells.")
else:
    print("Running locally — skipping Colab setup.")


In [ ]:
import json, os, re, sys, random
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"

ADAPTER_PATH = REPO_ROOT / "artifacts" / "models" / "qlora_v1"

LEARNING_RATE       = 5e-7
BATCH_SIZE          = 1
GRAD_ACCUM          = 4
EPOCHS              = 2
G                   = 8       # rollouts per question; A100 40GB handles G=8 comfortably
BETA                = 0.1
MAX_PROMPT_LENGTH   = 1024
MAX_COMPLETION_LEN  = 4096
ROLLOUT_TEMPERATURE = 0.7

LORA_RANK    = 32
LORA_ALPHA   = 64
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

TRAIN_SPLIT = 0.8
RANDOM_SEED = 42

PUBLIC_PATH = REPO_ROOT / "data" / "raw" / "public.jsonl"
GRPO_OUT    = REPO_ROOT / "artifacts" / "models" / "grpo_v1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# Detect execution environment
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# On Colab: redirect all paths to Google Drive
if IS_COLAB:
    DRIVE_BASE   = Path('/content/drive/MyDrive/math-qa-llm')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    PUBLIC_PATH  = DRIVE_BASE / "data" / "raw" / "public.jsonl"
    ADAPTER_PATH = DRIVE_BASE / "artifacts" / "models" / "qlora_v1"
    GRPO_OUT     = DRIVE_BASE / "artifacts" / "models" / "grpo_v1"

print(f"IS_COLAB     : {IS_COLAB}")
print(f"REPO_ROOT    : {REPO_ROOT}")
print(f"ADAPTER_PATH : {ADAPTER_PATH} | exists: {Path(ADAPTER_PATH).is_dir()}")
print(f"GRPO output  : {GRPO_OUT}")

In [ ]:
import subprocess, sys

for dep in ["peft", "trl>=0.12.0", "datasets", "accelerate"]:
    pkg = dep.split(">=")[0].split("==")[0]
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
from judger import Judger

judger_obj = Judger(strict_extract=False)

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load & Split Data

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You\'d better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You\'d better be sure of your answer."
)


def build_prompt(question: str, options) -> tuple:
    if options:
        labels   = [chr(65 + i) for i in range(len(options))]
        opts_txt = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_txt}"
    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint
    return SYSTEM_PROMPT_MATH, question


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

with open(PUBLIC_PATH, encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f]

rng = random.Random(RANDOM_SEED)
rng.shuffle(all_data)
n_train    = int(len(all_data) * TRAIN_SPLIT)
train_data = all_data[:n_train]
val_data   = all_data[n_train:]
print(f"Total: {len(all_data)} | Train: {len(train_data)} | Val: {len(val_data)}")


def make_dataset_row(item: dict) -> dict:
    """Convert public.jsonl item to GRPOTrainer row with a 'prompt' column."""
    system, user = build_prompt(item["question"], item.get("options"))
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        prompt_str = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        prompt_str = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
    return {
        "prompt":       prompt_str,
        "gold_answer":  json.dumps(item["answer"]),
        "options_json": json.dumps(item.get("options") or []),
        "item_id":      item["id"],
    }


train_dataset = Dataset.from_list([make_dataset_row(it) for it in train_data])
val_dataset   = Dataset.from_list([make_dataset_row(it) for it in val_data])

## 2. Reward Function

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def math_reward_fn(completions: list, gold_answer: list, options_json: list, **_) -> list:
    """Binary reward: 1.0 correct, 0.0 wrong. Called per micro-batch by GRPOTrainer."""
    rewards = []
    for pred, gold_str, opts_str in zip(completions, gold_answer, options_json):
        try:
            gold = json.loads(gold_str)
            opts = json.loads(opts_str)
            if isinstance(gold, str):
                correct = (extract_letter(pred) == gold.strip().upper())
            else:
                gold_list = gold if isinstance(gold, list) else [gold]
                correct   = judger_obj.auto_judge(
                    pred=pred, gold=gold_list, options=[[]] * len(gold_list)
                )
            rewards.append(1.0 if correct else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


_r = math_reward_fn(
    ["<think>\n1+1=2\n</think>\n\\boxed{2}", "<think>\n</think>\n\\boxed{3}"],
    [json.dumps(["2"]), json.dumps(["2"])],
    [json.dumps([]), json.dumps([])],
)
assert _r == [1.0, 0.0], f"Reward sanity check failed: {_r}"
print("Reward function OK.")

## 3. Load Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
base.config.use_cache = False

adapter_dir = Path(ADAPTER_PATH)
if adapter_dir.is_dir() and (adapter_dir / "adapter_config.json").is_file():
    model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    print(f"Loaded QLoRA adapter: {adapter_dir}")
else:
    lora_cfg = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)
    print("No adapter found — starting from base model with fresh LoRA.")

model.print_trainable_parameters()
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Train

In [ ]:
grpo_config = GRPOConfig(
    output_dir=str(GRPO_OUT / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    num_generations=G,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LEN,
    beta=BETA,
    temperature=ROLLOUT_TEMPERATURE,
    top_p=0.95,
    top_k=20,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[math_reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f"GRPO: {len(train_dataset)} train | G={G} | lr={LEARNING_RATE} | {EPOCHS} epochs")
trainer.train()
print("GRPO training complete.")

## 5. Quick Validation

In [ ]:
EVAL_N = 10
model.eval()

correct = 0
for i, row in enumerate(val_dataset.select(range(min(EVAL_N, len(val_dataset))))):
    inputs = tokenizer(
        row["prompt"], return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH
    ).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=512, temperature=0.6, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["gold_answer"])
    opts = json.loads(row["options_json"])
    try:
        if isinstance(gold, str):
            ok = (extract_letter(pred) == gold.strip().upper())
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            ok = judger_obj.auto_judge(pred=pred, gold=gold_list, options=[[]] * len(gold_list))
    except Exception:
        ok = False
    correct += int(ok)

print(f"Val accuracy (first {EVAL_N}): {correct}/{EVAL_N}")

## 6. Save Adapter

In [ ]:
GRPO_OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(GRPO_OUT))
tokenizer.save_pretrained(str(GRPO_OUT))
print(f"GRPO adapter saved: {GRPO_OUT}")
print("Set RUN_MERGE=True below, then run 05_private_submission.ipynb.")

## 7. Merge Adapter

In [ ]:
MERGE_OUTPUT = (DRIVE_BASE if IS_COLAB else REPO_ROOT) / "artifacts" / "models" / "grpo_v1_merged"
RUN_MERGE    = True

if RUN_MERGE:
    merged = model.merge_and_unload()
    MERGE_OUTPUT.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGE_OUTPUT), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGE_OUTPUT))
    print(f"Merged model saved: {MERGE_OUTPUT}")
else:
    print("Merge skipped (RUN_MERGE=False).")